# FLIR ADAS Thermal Dataset v2 — Preprocessing for Domain-Adaptive YOLOv8

This notebook preprocesses the **Teledyne FLIR Free ADAS Thermal Dataset v2** for training a **Domain-Adaptive Attentive YOLOv8** (DAAF-YOLOv8) model for real-time object detection in adverse weather conditions.

### Pipeline Overview
1. **Environment Setup** — Install dependencies, check GPU
2. **Dataset Download** — Pull dataset via `kagglehub`
3. **Dataset Exploration** — Inspect structure, class distribution, image statistics
4. **COCO → YOLO Conversion** — Convert JSON annotations to YOLO `.txt` format
5. **Thermal-Specific Preprocessing** — CLAHE contrast enhancement for thermal modality
6. **Adverse Weather Augmentation** — Fog/haze, rain, night-mode augmentations
7. **Dataset YAML Config** — Generate `data_rgb.yaml` and `data_thermal.yaml`
8. **Sanity Checks** — Verify label–image alignment, visualise samples

> **Dataset:** [Teledyne FLIR Free ADAS Thermal Dataset v2](https://www.kaggle.com/datasets/samdazel/teledyne-flir-adas-thermal-dataset-v2)  
> **Classes (16):** person, bike, car, motor, bus, train, truck, light, hydrant, sign, dog, deer, skateboard, stroller, scooter, other vehicle  
> **Frames:** 26 442 fully-annotated RGB + Thermal pairs

---
## 1. Environment Setup

In [ ]:
# Verify GPU availability
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU detected — consider enabling GPU in Runtime > Change runtime type')

In [ ]:
# Install required packages
!pip install -q kagglehub ultralytics albumentations opencv-python-headless tqdm pyyaml

In [ ]:
import os
import json
import shutil
import yaml
import random
import warnings
from glob import glob
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import albumentations as A

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

print('All imports successful.')

---
## 2. Dataset Download via KaggleHub

In [ ]:
# ── Option A: Kaggle API credentials via Google Drive (recommended for Colab) ──
# Uncomment and run if you have kaggle.json stored in your Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p ~/.kaggle
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# ── Option B: Enter credentials directly ──
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
# os.environ['KAGGLE_KEY']      = 'your_kaggle_api_key'

# ── Option C: Upload kaggle.json manually ──
# from google.colab import files
# files.upload()  # upload kaggle.json, then:
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

print('Configure ONE of the Kaggle auth options above, then run the download cell.')

In [ ]:
import kagglehub

# Download latest version of the FLIR ADAS Thermal Dataset v2
path = kagglehub.dataset_download('samdazel/teledyne-flir-adas-thermal-dataset-v2')
print('Path to dataset files:', path)

DATASET_ROOT = Path(path)
print('Dataset root:', DATASET_ROOT)

In [ ]:
# Inspect top-level dataset structure
print('Top-level contents:')
for item in sorted(DATASET_ROOT.iterdir()):
    print(f'  {item.name}/'  if item.is_dir() else f'  {item.name}')

# List all JSON annotation files
json_files = list(DATASET_ROOT.rglob('*.json'))
print(f'\nFound {len(json_files)} JSON annotation files:')
for jf in json_files:
    print(f'  {jf.relative_to(DATASET_ROOT)}')

In [ ]:
# Auto-detect dataset splits and modalities
# The FLIR v2 dataset uses the structure:
#   <split>/  ->  images_rgb/  images_thermal/  coco.json  (or separate jsons)

SPLITS = {}

# Common split name patterns
SPLIT_KEYWORDS = {
    'train': ['train', 'images_rgb_train', 'images_thermal_train'],
    'val':   ['val', 'images_rgb_val', 'images_thermal_val'],
    'test':  ['test', 'video_rgb_test', 'video_thermal_test'],
}

# Collect all image directories
all_img_dirs = [d for d in DATASET_ROOT.rglob('*') if d.is_dir() and
                any(d.glob('*.jpg')) or any(d.glob('*.png'))]

print('Image directories found:')
for d in sorted(all_img_dirs):
    n_jpg = len(list(d.glob('*.jpg')))
    n_png = len(list(d.glob('*.png')))
    print(f'  {d.relative_to(DATASET_ROOT)}  ->  {n_jpg} jpg, {n_png} png')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# OUTPUT DIRECTORY CONFIGURATION
# Adjust the paths below to match the actual downloaded structure.
# ─────────────────────────────────────────────────────────────────

OUTPUT_BASE = Path('/content/datasets')

# Map: output_dir_name -> (images_src_dir, coco_json_src)
# Edit these to match what was printed in the cell above.
DATASET_MAP = {
    'images_rgb_train':     (DATASET_ROOT / 'images_rgb_train'    / 'images', DATASET_ROOT / 'images_rgb_train'    / 'coco.json'),
    'images_thermal_train': (DATASET_ROOT / 'images_thermal_train'/ 'images', DATASET_ROOT / 'images_thermal_train'/ 'coco.json'),
    'images_rgb_val':       (DATASET_ROOT / 'images_rgb_val'      / 'images', DATASET_ROOT / 'images_rgb_val'      / 'coco.json'),
    'images_thermal_val':   (DATASET_ROOT / 'images_thermal_val'  / 'images', DATASET_ROOT / 'images_thermal_val'  / 'coco.json'),
    'video_rgb_test':       (DATASET_ROOT / 'video_rgb_test'      / 'images', DATASET_ROOT / 'video_rgb_test'      / 'coco.json'),
    'video_thermal_test':   (DATASET_ROOT / 'video_thermal_test'  / 'images', DATASET_ROOT / 'video_thermal_test'  / 'coco.json'),
}

# Class names (FLIR v2 — 16 classes, IDs 0–15)
CLASS_NAMES = [
    'person', 'bike', 'car', 'motor', 'bus', 'train', 'truck',
    'light', 'hydrant', 'sign', 'dog', 'deer', 'skateboard',
    'stroller', 'scooter', 'other vehicle'
]

print('Output base:', OUTPUT_BASE)
print(f'Managing {len(DATASET_MAP)} splits.')

---
## 3. Dataset Exploration

In [ ]:
def load_coco_json(json_path):
    """Load a COCO-format annotation JSON and return the parsed dict."""
    with open(json_path) as f:
        return json.load(f)


def summarise_split(name, img_dir, json_path):
    """Print a quick summary of one dataset split."""
    images = list(Path(img_dir).glob('*.jpg')) + list(Path(img_dir).glob('*.png'))
    data   = load_coco_json(json_path)
    cats   = {c['id']: c['name'] for c in data['categories']}
    counts = Counter(a['category_id'] for a in data['annotations'])

    print(f'\n── {name} ──')
    print(f'  Images      : {len(images)}')
    print(f'  Annotations : {len(data["annotations"])}')
    print(f'  Classes     : {len(cats)}')
    print('  Top-5 classes by annotation count:')
    for cid, cnt in sorted(counts.items(), key=lambda x: -x[1])[:5]:
        print(f'    [{cid:2d}] {cats.get(cid, "?"):<20s} {cnt}')
    return counts


all_counts = {}
for split_name, (img_dir, json_path) in DATASET_MAP.items():
    if json_path.exists() and img_dir.exists():
        all_counts[split_name] = summarise_split(split_name, img_dir, json_path)
    else:
        print(f'\n[WARN] Missing: {split_name}  (check DATASET_MAP paths)')

In [ ]:
# Class-distribution bar chart (training thermal split)
train_thermal_counts = all_counts.get('images_thermal_train', {})

if train_thermal_counts:
    fig, ax = plt.subplots(figsize=(14, 5))
    ids   = sorted(train_thermal_counts.keys())
    vals  = [train_thermal_counts[i] for i in ids]
    names = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i) for i in ids]

    bars = ax.bar(names, vals, color='steelblue', edgecolor='white')
    ax.set_title('Annotation Count per Class — Thermal Train Split', fontsize=14)
    ax.set_xlabel('Class'); ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(v), ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
def show_samples(img_dir, title='', n=8, cmap=None):
    """Display a grid of random sample images from img_dir."""
    paths = list(Path(img_dir).glob('*.jpg')) + list(Path(img_dir).glob('*.png'))
    if not paths:
        print(f'No images found in {img_dir}'); return

    sample = random.sample(paths, min(n, len(paths)))
    cols = 4; rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    fig.suptitle(title, fontsize=14)
    axes = axes.flatten()

    for ax, p in zip(axes, sample):
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else np.zeros((100,100,3), dtype=np.uint8)
        ax.imshow(img, cmap=cmap)
        ax.set_title(p.stem[-20:], fontsize=7)
        ax.axis('off')

    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()


# RGB training samples
show_samples(DATASET_MAP['images_rgb_train'][0],     title='RGB Training Samples')
# Thermal training samples
show_samples(DATASET_MAP['images_thermal_train'][0], title='Thermal Training Samples', cmap='inferno')

In [ ]:
def compute_image_stats(img_dir, modality='rgb', n_samples=500):
    """
    Compute mean/std pixel statistics on a random sample.
    Returns per-channel mean and std normalised to [0,1].
    """
    paths = list(Path(img_dir).glob('*.jpg')) + list(Path(img_dir).glob('*.png'))
    sample = random.sample(paths, min(n_samples, len(paths)))

    means, stds = [], []
    for p in tqdm(sample, desc=f'Stats [{modality}]'):
        img = cv2.imread(str(p)).astype(np.float32) / 255.0
        if img is None:
            continue
        means.append(img.mean(axis=(0, 1)))
        stds.append(img.std(axis=(0, 1)))

    mean = np.mean(means, axis=0)[::-1]  # BGR->RGB
    std  = np.mean(stds,  axis=0)[::-1]
    print(f'  [{modality}] mean={mean.round(4)}, std={std.round(4)}')
    return mean, std


print('Computing image statistics (used for normalisation):')
rgb_mean, rgb_std         = compute_image_stats(DATASET_MAP['images_rgb_train'][0],     'rgb')
thermal_mean, thermal_std = compute_image_stats(DATASET_MAP['images_thermal_train'][0], 'thermal')

---
## 4. COCO JSON → YOLO TXT Conversion

In [ ]:
def make_dir(path, clean=False):
    """Create directory, optionally removing it first."""
    path = Path(path)
    if clean and path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def coco_bbox_to_yolo(img_w, img_h, bbox):
    """
    Convert COCO bbox [x_tl, y_tl, w, h] to YOLO format
    [x_center_rel, y_center_rel, w_rel, h_rel].

    Clips values to [0, 1] to guard against annotation noise.
    """
    x_tl, y_tl, w, h = bbox
    x_c = np.clip((x_tl + w / 2.0) / img_w, 0.0, 1.0)
    y_c = np.clip((y_tl + h / 2.0) / img_h, 0.0, 1.0)
    w_r = np.clip(w / img_w, 0.0, 1.0)
    h_r = np.clip(h / img_h, 0.0, 1.0)
    return x_c, y_c, w_r, h_r


def is_valid_bbox(x_c, y_c, w_r, h_r, min_size=1e-4):
    """Filter out degenerate boxes (zero-area or out-of-range)."""
    return (0 < w_r <= 1.0) and (0 < h_r <= 1.0) and \
           (0 <= x_c <= 1.0) and (0 <= y_c <= 1.0) and \
           (w_r >= min_size) and (h_r >= min_size)


def convert_coco_json_to_yolo_txt(output_label_dir, json_file,
                                   class_names=None, remap_ids=True):
    """
    Convert COCO-format annotations to individual YOLO .txt files.

    Parameters
    ----------
    output_label_dir : str | Path  — directory where .txt files are written
    json_file        : str | Path  — path to the COCO coco.json file
    class_names      : list[str]   — canonical ordered class names
    remap_ids        : bool        — remap sparse category IDs to 0-based contiguous

    Notes
    -----
    The FLIR v2 dataset ships with 16 category IDs that are NOT necessarily
    contiguous (e.g. some IDs from the 80-class COCO set are skipped).
    Setting remap_ids=True remaps them to 0–15 in the order they appear
    in the JSON `categories` list.
    """
    out_dir = make_dir(output_label_dir, clean=True)

    with open(json_file) as f:
        data = json.load(f)

    # Build category mapping
    cats_ordered = sorted(data['categories'], key=lambda c: c['id'])
    if remap_ids:
        # Map original ID -> 0-based index
        id_map = {c['id']: i for i, c in enumerate(cats_ordered)}
    else:
        id_map = {c['id']: c['id'] for c in cats_ordered}

    # Write class name file
    names = class_names or [c['name'] for c in cats_ordered]
    with open(out_dir / '_darknet.labels', 'w') as f:
        f.write('\n'.join(names) + '\n')

    # Index annotations by image_id for O(1) lookup
    ann_index = {}
    for ann in data['annotations']:
        ann_index.setdefault(ann['image_id'], []).append(ann)

    skipped_boxes = 0
    for img_info in tqdm(data['images'], desc=f'Converting {Path(json_file).parent.name}'):
        img_id  = img_info['id']
        img_w   = img_info['width']
        img_h   = img_info['height']
        # Strip any leading 'data/' prefix from filenames
        stem    = Path(img_info['file_name']).stem

        txt_path = out_dir / f'{stem}.txt'
        anns     = ann_index.get(img_id, [])

        with open(txt_path, 'w') as f:
            for ann in anns:
                cat_id = id_map.get(ann['category_id'])
                if cat_id is None:
                    continue
                x_c, y_c, w_r, h_r = coco_bbox_to_yolo(img_w, img_h, ann['bbox'])
                if not is_valid_bbox(x_c, y_c, w_r, h_r):
                    skipped_boxes += 1
                    continue
                f.write(f'{cat_id} {x_c:.6f} {y_c:.6f} {w_r:.6f} {h_r:.6f}\n')

    print(f'  Done. Skipped {skipped_boxes} degenerate boxes.')


print('Conversion utilities defined.')

In [ ]:
def copy_images(src_dir, dst_dir):
    """Copy all jpg/png images from src_dir to dst_dir."""
    dst = make_dir(dst_dir, clean=True)
    images = list(Path(src_dir).glob('*.jpg')) + list(Path(src_dir).glob('*.png'))
    for img in tqdm(images, desc=f'Copying images -> {Path(dst_dir).name}'):
        shutil.copy2(img, dst / img.name)
    print(f'  Copied {len(images)} images to {dst}')
    return len(images)


print('Image copy utility defined.')

In [ ]:
# ── Run conversion for all splits ──────────────────────────────────────────────
for split_name, (img_src, json_src) in DATASET_MAP.items():
    if not json_src.exists():
        print(f'[SKIP] {split_name}: JSON not found at {json_src}')
        continue
    if not img_src.exists():
        print(f'[SKIP] {split_name}: image dir not found at {img_src}')
        continue

    print(f'\n=== Processing: {split_name} ===')

    dst_img_dir   = OUTPUT_BASE / split_name / 'images'
    dst_label_dir = OUTPUT_BASE / split_name / 'labels'

    copy_images(img_src, dst_img_dir)
    convert_coco_json_to_yolo_txt(
        output_label_dir=dst_label_dir,
        json_file=json_src,
        class_names=CLASS_NAMES,
        remap_ids=True,
    )

print('\n=== All splits converted ===')

---
## 5. Thermal-Specific Preprocessing — CLAHE Enhancement

Thermal cameras produce 8-bit grayscale-mapped images with low global contrast.  
**CLAHE** (Contrast Limited Adaptive Histogram Equalisation) improves local contrast without over-amplifying noise — critical for a domain-adaptive detector.

In [ ]:
def apply_clahe(img_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Apply CLAHE to the L channel of an LAB image.
    Works on both grayscale-mapped thermal images and true RGB.
    """
    lab   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    lab[..., 0] = clahe.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


# Visualise CLAHE effect on a thermal sample
thermal_imgs = list((OUTPUT_BASE / 'images_thermal_train' / 'images').glob('*.jpg'))
if thermal_imgs:
    sample_path = random.choice(thermal_imgs)
    raw  = cv2.imread(str(sample_path))
    enhanced = apply_clahe(raw)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(raw,      cv2.COLOR_BGR2RGB), cmap='inferno'); axes[0].set_title('Original Thermal');  axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB), cmap='inferno'); axes[1].set_title('CLAHE Enhanced');   axes[1].axis('off')
    plt.suptitle('CLAHE Contrast Enhancement for Thermal Modality', fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('No thermal images found — run the conversion cell first.')

In [ ]:
def enhance_thermal_split(split_name, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Apply CLAHE in-place to all thermal images in a split.
    The enhanced images overwrite the originals in the output directory.
    """
    img_dir = OUTPUT_BASE / split_name / 'images'
    paths   = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    if not paths:
        print(f'[SKIP] No images in {img_dir}'); return

    for p in tqdm(paths, desc=f'CLAHE [{split_name}]'):
        img = cv2.imread(str(p))
        if img is None:
            continue
        enhanced = apply_clahe(img, clip_limit=clip_limit, tile_grid=tile_grid)
        cv2.imwrite(str(p), enhanced)

    print(f'  Enhanced {len(paths)} images.')


# Apply to all thermal splits
for split in ['images_thermal_train', 'images_thermal_val', 'video_thermal_test']:
    enhance_thermal_split(split)

---
## 6. Adverse Weather Augmentation (Domain Adaptation)

To train a model that is robust to **fog, haze, rain, and low-light** conditions we apply physics-inspired augmentations using Albumentations during preprocessing (offline augmentation).  
This doubles/triples the effective training set for rare adverse-weather scenarios.

In [ ]:
# ── Augmentation pipelines ─────────────────────────────────────────────────────

FOG_AUG = A.Compose([
    A.RandomFog(fog_coef_lower=0.3, fog_coef_upper=0.6,
                alpha_coef=0.08, p=1.0),
    A.GaussianBlur(blur_limit=(3, 5), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=(-0.15, 0.05),
                               contrast_limit=(-0.2, 0.0), p=0.8),
], bbox_params=A.BboxParams(format='yolo', label_fields=['labels'], min_visibility=0.2))

RAIN_AUG = A.Compose([
    A.RandomRain(slant_lower=-10, slant_upper=10,
                 drop_length=15, drop_width=1,
                 drop_color=(180, 180, 180), blur_value=3,
                 brightness_coefficient=0.85, rain_type='drizzle', p=1.0),
    A.GaussNoise(var_limit=(10, 30), p=0.4),
], bbox_params=A.BboxParams(format='yolo', label_fields=['labels'], min_visibility=0.2))

NIGHT_AUG = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.5, -0.3),
                               contrast_limit=(-0.1, 0.1), p=1.0),
    A.GaussNoise(var_limit=(20, 50), p=0.6),
    A.RandomGamma(gamma_limit=(40, 70), p=0.5),
], bbox_params=A.BboxParams(format='yolo', label_fields=['labels'], min_visibility=0.2))


AUG_PIPELINES = {
    'fog':   FOG_AUG,
    'rain':  RAIN_AUG,
    'night': NIGHT_AUG,
}

print('Augmentation pipelines defined:', list(AUG_PIPELINES.keys()))

In [ ]:
def read_yolo_labels(label_path):
    """Read YOLO label file; return (class_ids, bboxes) as lists."""
    class_ids, bboxes = [], []
    if not Path(label_path).exists():
        return class_ids, bboxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                class_ids.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:]])
    return class_ids, bboxes


def write_yolo_labels(label_path, class_ids, bboxes):
    """Write YOLO label file."""
    with open(label_path, 'w') as f:
        for cid, bbox in zip(class_ids, bboxes):
            f.write(f'{cid} {" ".join(f"{v:.6f}" for v in bbox)}\n')


def augment_and_save(img_path, label_path, aug_pipeline, aug_name,
                     out_img_dir, out_label_dir):
    """
    Apply one augmentation pipeline to an image+label pair and save
    the result with a suffix indicating the augmentation type.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    class_ids, bboxes = read_yolo_labels(label_path)
    if not bboxes:
        return  # skip unannotated images for augmentation

    try:
        result = aug_pipeline(image=img_rgb, bboxes=bboxes, labels=class_ids)
    except Exception:
        return

    if not result['bboxes']:
        return  # all boxes were filtered out

    stem    = Path(img_path).stem
    ext     = Path(img_path).suffix
    new_stem = f'{stem}_{aug_name}'

    aug_img = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(Path(out_img_dir) / f'{new_stem}{ext}'), aug_img)
    write_yolo_labels(
        Path(out_label_dir) / f'{new_stem}.txt',
        result['labels'], result['bboxes']
    )


print('Augmentation helpers defined.')

In [ ]:
# ── Preview augmentations on one sample image ──────────────────────────────────
rgb_imgs = list((OUTPUT_BASE / 'images_rgb_train' / 'images').glob('*.jpg'))
if rgb_imgs:
    sample_img_path   = random.choice(rgb_imgs)
    sample_label_path = OUTPUT_BASE / 'images_rgb_train' / 'labels' / (sample_img_path.stem + '.txt')

    raw     = cv2.cvtColor(cv2.imread(str(sample_img_path)), cv2.COLOR_BGR2RGB)
    cids, bboxes = read_yolo_labels(sample_label_path)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('Adverse Weather Augmentations Preview', fontsize=14)

    axes[0].imshow(raw); axes[0].set_title('Original'); axes[0].axis('off')

    for ax, (aug_name, aug_fn) in zip(axes[1:], AUG_PIPELINES.items()):
        if bboxes:
            try:
                res = aug_fn(image=raw, bboxes=bboxes, labels=cids)
                aug_img = res['image']
            except Exception:
                aug_img = raw
        else:
            aug_img = raw
        ax.imshow(aug_img)
        ax.set_title(aug_name.capitalize())
        ax.axis('off')

    plt.tight_layout(); plt.show()
else:
    print('No RGB images found — run conversion first.')

In [ ]:
# ── Apply augmentations to RGB training split only ─────────────────────────────
# (Thermal augmentation is handled separately; fog/rain physics differ for IR)

AUG_SPLIT     = 'images_rgb_train'
MAX_AUG_IMGS  = 2000   # cap to avoid very long runtimes; set to None for all

img_dir   = OUTPUT_BASE / AUG_SPLIT / 'images'
label_dir = OUTPUT_BASE / AUG_SPLIT / 'labels'
all_imgs  = list(img_dir.glob('*.jpg'))

if MAX_AUG_IMGS:
    aug_imgs = random.sample(all_imgs, min(MAX_AUG_IMGS, len(all_imgs)))
else:
    aug_imgs = all_imgs

print(f'Augmenting {len(aug_imgs)} images with {len(AUG_PIPELINES)} pipelines...')

for aug_name, aug_pipeline in AUG_PIPELINES.items():
    print(f'  [{aug_name}]')
    for img_path in tqdm(aug_imgs, desc=aug_name):
        label_path = label_dir / (img_path.stem + '.txt')
        augment_and_save(
            img_path=img_path,
            label_path=label_path,
            aug_pipeline=aug_pipeline,
            aug_name=aug_name,
            out_img_dir=img_dir,
            out_label_dir=label_dir,
        )

n_final = len(list(img_dir.glob('*.jpg')))
print(f'\nTraining set size after augmentation: {n_final} images')

---
## 7. Dataset YAML Configuration Files

In [ ]:
CONFIG_DIR = Path('/content/config')
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def write_yaml(path, config_dict):
    with open(path, 'w') as f:
        yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)
    print(f'Written: {path}')


# ── Thermal YAML ───────────────────────────────────────────────────────────────
thermal_cfg = {
    'path' : str(OUTPUT_BASE),
    'train': 'images_thermal_train/images',
    'val'  : 'images_thermal_val/images',
    'test' : 'video_thermal_test/images',
    'nc'   : len(CLASS_NAMES),
    'names': CLASS_NAMES,
}
write_yaml(CONFIG_DIR / 'data_thermal.yaml', thermal_cfg)

# ── RGB YAML ───────────────────────────────────────────────────────────────────
rgb_cfg = {
    'path' : str(OUTPUT_BASE),
    'train': 'images_rgb_train/images',
    'val'  : 'images_rgb_val/images',
    'test' : 'video_rgb_test/images',
    'nc'   : len(CLASS_NAMES),
    'names': CLASS_NAMES,
}
write_yaml(CONFIG_DIR / 'data_rgb.yaml', rgb_cfg)

# ── Print contents ─────────────────────────────────────────────────────────────
print('\n── data_thermal.yaml ──')
print(open(CONFIG_DIR / 'data_thermal.yaml').read())
print('── data_rgb.yaml ──')
print(open(CONFIG_DIR / 'data_rgb.yaml').read())

---
## 8. Sanity Checks & Visualisation

In [ ]:
def sanity_check_split(split_name):
    """
    For each split, verify:
      - Image count matches label count
      - No empty label files (images with no annotations)
      - All bbox values are in [0, 1]
    """
    img_dir   = OUTPUT_BASE / split_name / 'images'
    label_dir = OUTPUT_BASE / split_name / 'labels'

    imgs   = set(p.stem for p in img_dir.glob('*.jpg'))
    labels = set(p.stem for p in label_dir.glob('*.txt')
                 if not p.name.startswith('_'))

    imgs_no_label = imgs - labels
    labels_no_img = labels - imgs

    out_of_range = 0
    for lp in label_dir.glob('*.txt'):
        if lp.name.startswith('_'):
            continue
        with open(lp) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    vals = [float(v) for v in parts[1:]]
                    if any(v < 0 or v > 1 for v in vals):
                        out_of_range += 1

    print(f'\n── {split_name} ──')
    print(f'  Images : {len(imgs)}')
    print(f'  Labels : {len(labels)}')
    print(f'  Images without label : {len(imgs_no_label)}')
    print(f'  Labels without image : {len(labels_no_img)}')
    print(f'  Out-of-range bbox values : {out_of_range}')
    ok = len(imgs_no_label) == 0 and len(labels_no_img) == 0 and out_of_range == 0
    print(f'  Status: {"OK" if ok else "WARNINGS FOUND"}')


for split in DATASET_MAP:
    sanity_check_split(split)

In [ ]:
def draw_yolo_boxes(img_rgb, label_path, class_names, color=(0, 255, 0)):
    """Draw YOLO bounding boxes onto an RGB image (returns annotated copy)."""
    img = img_rgb.copy()
    h, w = img.shape[:2]
    cids, bboxes = read_yolo_labels(label_path)
    for cid, (xc, yc, bw, bh) in zip(cids, bboxes):
        x1 = int((xc - bw / 2) * w); y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w); y2 = int((yc + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = class_names[cid] if cid < len(class_names) else str(cid)
        cv2.putText(img, label, (x1, max(y1 - 4, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return img


def show_annotated_samples(split_name, n=6, cmap=None, color=(0, 255, 0)):
    img_dir   = OUTPUT_BASE / split_name / 'images'
    label_dir = OUTPUT_BASE / split_name / 'labels'

    paths = [p for p in img_dir.glob('*.jpg')
             if (label_dir / (p.stem + '.txt')).exists()]
    if not paths:
        print(f'No annotated images in {split_name}'); return

    sample = random.sample(paths, min(n, len(paths)))
    cols = 3; rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
    fig.suptitle(f'Annotated Samples — {split_name}', fontsize=14)
    axes = axes.flatten()

    for ax, p in zip(axes, sample):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        label_path = label_dir / (p.stem + '.txt')
        annotated  = draw_yolo_boxes(img, label_path, CLASS_NAMES, color)
        ax.imshow(annotated, cmap=cmap); ax.set_title(p.stem[-18:], fontsize=7); ax.axis('off')

    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()


show_annotated_samples('images_rgb_train',     color=(0, 220, 0))
show_annotated_samples('images_thermal_train', color=(255, 100, 0), cmap='inferno')

In [ ]:
# ── Final dataset summary ──────────────────────────────────────────────────────
print('='*60)
print('PREPROCESSING COMPLETE — Dataset Summary')
print('='*60)
for split_name in DATASET_MAP:
    img_dir = OUTPUT_BASE / split_name / 'images'
    if img_dir.exists():
        n = len(list(img_dir.glob('*.jpg')))
        print(f'  {split_name:<30s}: {n:>6d} images')

print(f'\n  Classes : {len(CLASS_NAMES)}')
print(f'  Config  : {CONFIG_DIR}/data_thermal.yaml')
print(f'            {CONFIG_DIR}/data_rgb.yaml')
print('\nReady for training with: model.train(data="/content/config/data_thermal.yaml", ...)')

---
## Next Steps

```python
from ultralytics import YOLO

# Train on thermal modality
model = YOLO('yolov8s.pt')
model.train(
    data   = '/content/config/data_thermal.yaml',
    epochs = 100,
    imgsz  = 640,
    batch  = 16,
    device = 0,          # GPU
    name   = 'yolov8s_thermal',
    mosaic = 1.0,        # mosaic augmentation
    degrees= 5.0,        # rotation
    fliplr = 0.5,
)

# Train on RGB modality
model_rgb = YOLO('yolov8s.pt')
model_rgb.train(
    data   = '/content/config/data_rgb.yaml',
    epochs = 100,
    imgsz  = 640,
    batch  = 16,
    device = 0,
    name   = 'yolov8s_rgb',
)
```

See `02_Model_Training.ipynb` for the full Domain-Adaptive Attentive Fusion (DAAF) training pipeline.